## Import and Setup

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from tdc.multi_pred import DrugRes
from rdkit import Chem
from rdkit.Chem import AllChem
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error
import gc

## Load data

In [2]:
data = DrugRes(name = 'GDSC2', path='../data')

Found local copy...
Loading...
Done!


In [3]:
df = data.get_data()
df.head()

,Drug_ID,Drug,Cell Line_ID,Cell Line,Y
0,Camptothecin,CC[C@@]1(C2=C(COC1=O)C(=O)N3CC4=CC5=CC=CC=C5N=...,HCC1954,"[8.54820830373167, 2.5996072676336297, 10.3759...",-0.251083
1,Camptothecin,CC[C@@]1(C2=C(COC1=O)C(=O)N3CC4=CC5=CC=CC=C5N=...,HCC1143,"[7.58193774904993, 2.81430257671695, 10.363326...",1.343315
2,Camptothecin,CC[C@@]1(C2=C(COC1=O)C(=O)N3CC4=CC5=CC=CC=C5N=...,HCC1187,"[9.013252540641961, 2.9520929896608, 9.3474286...",1.736985
3,Camptothecin,CC[C@@]1(C2=C(COC1=O)C(=O)N3CC4=CC5=CC=CC=C5N=...,HCC1395,"[7.4351511634642105, 2.8325700611437004, 10.34...",-2.309078
4,Camptothecin,CC[C@@]1(C2=C(COC1=O)C(=O)N3CC4=CC5=CC=CC=C5N=...,HCC1599,"[8.334239608034789, 2.7477031637484997, 10.314...",-3.106684


# Preprocess

In [4]:
# convert SMILES to Fingerprint
def smiles_to_fp(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return np.zeros(1024, dtype=np.int8)

        return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024), dtype=np.int8)
    except:
        return np.zeros(1024, dtype=np.int8)

In [5]:
drug_features = np.array([smiles_to_fp(s) for s in df['Drug'].values], dtype=np.int8)

In [6]:
drug_features.shape

(92703, 1024)

In [7]:
num_samples = len(df)
num_cell_feats = len(df['Cell Line'].iloc[0])

cell_features = np.zeros((num_samples, num_cell_feats), dtype=np.float32)

for i, val in enumerate(df['Cell Line'].values):
    cell_features[i] = np.array(val, dtype=np.float32)

In [8]:
cell_features.shape

(92703, 17737)

In [9]:
df.drop(columns=['Cell Line'], inplace=True)
gc.collect() 

133

# Train/Val/Test split

In [10]:
X = np.hstack([drug_features, cell_features])
y = df["Y"].values

del drug_features
del cell_features
gc.collect()

0

In [11]:
split = data.get_split(method='random', seed=42, frac=[0.7, 0.1, 0.2])

train_idx = split['train'].index
val_idx = split['valid'].index
test_idx = split['test'].index

In [12]:
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

In [13]:
del X
gc.collect()

0

# Train LightGBM

In [14]:
model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=31,
    n_jobs=-1,
    importance_type='split',
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=20)
    ]
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 8.315027 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4512964
[LightGBM] [Info] Number of data points in the train set: 64892, number of used features: 18619
[LightGBM] [Info] Start training from score 2.579641
Training until validation scores don't improve for 50 rounds
[20]	valid_0's rmse: 2.02172	valid_0's l2: 4.08734
[40]	valid_0's rmse: 1.57964	valid_0's l2: 2.49525
[60]	valid_0's rmse: 1.3947	valid_0's l2: 1.94518
[80]	valid_0's rmse: 1.29605	valid_0's l2: 1.67975
[100]	valid_0's rmse: 1.23164	valid_0's l2: 1.51694
[120]	valid_0's rmse: 1.18267	valid_0's l2: 1.39872
[140]	valid_0's rmse: 1.14849	valid_0's l2: 1.31902
[160]	valid_0's rmse: 1.12014	valid_0's l2: 1.25472
[180]	valid_0's rmse: 1.09891	valid_0's l2: 1.20761
[200]	valid_0's rmse: 1.08332	valid_0's l2: 1.17358
[220]	valid_0's rmse: 1.0682	valid_0's l2: 1.14105
[240]	valid_0's rmse: 1.0

LGBMRegressor(learning_rate=0.05, n_estimators=2000, n_jobs=-1, random_state=42)

# Evaluate

In [15]:
print("--- 6. Đang đánh giá trên tập Test ---")
y_pred = model.predict(X_test)

p_corr, _ = pearsonr(y_test, y_pred)
s_corr, _ = spearmanr(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n" + "="*30)
print("KẾT QUẢ DỰ ĐOÁN (LGBM BASELINE)")
print(f"Pearson Correlation:  {p_corr:.4f}")
print(f"Spearman Correlation: {s_corr:.4f}")
print(f"RMSE:                {rmse:.4f}")
print("="*30)

--- 6. Đang đánh giá trên tập Test ---

KẾT QUẢ DỰ ĐOÁN (LGBM BASELINE)
Pearson Correlation:  0.9812
Spearman Correlation: 0.9746
RMSE:                0.5809


In [18]:
import joblib
import os

# Tạo thư mục models nếu chưa có
os.makedirs('../models', exist_ok=True)

# Lưu mô hình
model_path = '../models/lgbm_drp_baseline.pkl'
joblib.dump(model, model_path)
print(f"Đã lưu mô hình tại: {model_path}")

Đã lưu mô hình tại: ../models/lgbm_drp_baseline.pkl
